# 04 — Behavior Embedding

Feature extraction → 50 ghost users → UMAP (n=54) → HDBSCAN → keep only 4 real personas.
Output: `output/embeddings.json` with {x, y, cluster_id} per persona.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from embedder import extract_features, embed_personas
from pattern_miner import token_entropy

OUTPUT = Path('..') / 'output'
PERSONAS = ['aisyah', 'daniel', 'mei_ling', 'hafiz']

In [ ]:
# Load tokens and motif scores
real_features = {}

for persona in PERSONAS:
    with open(OUTPUT / f'tokens_{persona}.json', encoding='utf-8') as f:
        tokens = json.load(f)
    with open(OUTPUT / f'motifs_{persona}.json', encoding='utf-8') as f:
        motif_data = json.load(f)

    motif_strength = motif_data['motif_strength_score']
    h = token_entropy(tokens)
    fvec = extract_features(tokens, motif_strength, h)
    real_features[persona] = fvec
    print(f'{persona:<12}  tokens={len(tokens)}  motif={motif_strength}  H={h:.3f}  feat_dim={fvec.shape[0]}')

In [ ]:
# UMAP + HDBSCAN with 50 ghost users
results = embed_personas(real_features, n_ghosts=50, seed=42)

print(f'\n{"Persona":<12}  {"x":>8}  {"y":>8}  {"cluster":>8}')
print('-' * 44)
for persona, data in results.items():
    print(f'{persona:<12}  {data["x"]:>8.4f}  {data["y"]:>8.4f}  {data["cluster_id"]:>8}')

In [ ]:
# Write output
out_path = OUTPUT / 'embeddings.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(f'Written: {out_path}')